# SE3082 Lab 10 - CUDA timing, checks, and multidimensional grids

**Student:** IT24101492

Run this notebook in Google Colab with a GPU runtime. It contains the source files from this repository as `%%writefile` cells, so it can recreate them on Colab’s temporary disk. There are no saved GPU outputs in this file; run the cells there to record real timings and output.

The compile cells use `-arch=sm_75` for a Tesla T4. Check the GPU compute capability below and change the flag if Colab assigns a different GPU.

## Before you begin - Colab setup

Choose **Runtime → Change runtime type → T4 GPU** (or the available GPU), then run both checks.

In [ ]:
!nvidia-smi
!nvidia-smi --query-gpu=name,compute_cap --format=csv

In [ ]:
!nvcc --version

## Exercise 1 - Time a GPU program

The timer starts immediately before the kernel and stops after `cudaDeviceSynchronize()`. The synchronization matters because kernel launches are asynchronous: without it, the CPU clock can stop while the GPU is still working.

In [ ]:
%%writefile ex1_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(const int *a, const int *b, int *c)
{
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

static double cpuSecond()
{
    timeval time_value{};
    gettimeofday(&time_value, nullptr);
    return static_cast<double>(time_value.tv_sec) + time_value.tv_usec * 1.0e-6;
}

static void random_ints(int *values, int count)
{
    for (int i = 0; i < count; ++i)
        values[i] = std::rand() % 100;
}

int main()
{
    const size_t bytes = static_cast<size_t>(N) * sizeof(int);
    int *a = static_cast<int *>(std::malloc(bytes));
    int *b = static_cast<int *>(std::malloc(bytes));
    int *c = static_cast<int *>(std::malloc(bytes));
    int *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    if (!a || !b || !c) {
        std::fprintf(stderr, "Could not allocate host arrays.\n");
        return 1;
    }
    random_ints(a, N);
    random_ints(b, N);
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);
    cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice);

    const double started = cpuSecond();
#ifdef EXPERIMENT_BAD_LAUNCH
    add<<<N, 2048>>>(d_a, d_b, d_c);
#else
    add<<<N, 1>>>(d_a, d_b, d_c);
#endif
#ifndef OMIT_DEVICE_SYNC
    cudaDeviceSynchronize();
#endif
    const double elapsed = cpuSecond() - started;

    // This copy waits for the GPU even in the no-synchronize experiment.
    cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost);
    for (int i = 0; i < 5; ++i)
        std::printf("%d + %d = %d\n", a[i], b[i], c[i]);
    int errors = 0;
    for (int i = 0; i < N; ++i)
        if (c[i] != a[i] + b[i])
            ++errors;
    std::printf("Verification: %d mismatches out of %d\n", errors, N);
    std::printf("Time taken - %.7f\n", elapsed);

    std::free(a); std::free(b); std::free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


In [ ]:
!nvcc -arch=sm_75  ex1_timed.cu -o ex1_timed && ./ex1_timed

### Synchronization experiment

The next cell rebuilds the same source with synchronization omitted. The measured time should be much smaller because it mostly measures the launch. The later device-to-host copy waits for the GPU, so the result check can still pass. Rebuild the normal executable with the preceding cell afterward.

In [ ]:
!nvcc -arch=sm_75 -DOMIT_DEVICE_SYNC ex1_timed.cu -o ex1_no_sync && ./ex1_no_sync

### Time the Lab 9 kernels (three runs each)

These are timed copies of Lab 9 Exercises 3, 4, 5, and 6. Each program prints `Time taken` for the kernel only, with synchronization before the timer stops. Compile the four programs first, then run the next cell to capture three measurements per program into a table and `lab9_timing_results.csv`. Array initialization and copies are outside these kernel-only times.

In [ ]:
%%writefile ex1_lab9_ex3_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(const int *a, const int *b, int *c)
{
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

static bool check_cuda(cudaError_t error, const char *operation)
{
    if (error == cudaSuccess)
        return true;
    std::fprintf(stderr, "%s failed: %s\n", operation, cudaGetErrorString(error));
    return false;
}

static void random_ints(int *values, int count)
{
    for (int i = 0; i < count; ++i)
        values[i] = std::rand() % 100;
}

static double cpuSecond()
{
    timeval time_value{};
    gettimeofday(&time_value, nullptr);
    return static_cast<double>(time_value.tv_sec) + time_value.tv_usec * 1.0e-6;
}

int main()
{
    double dStart = 0.0, dElaps = 0.0;
    const size_t bytes = static_cast<size_t>(N) * sizeof(int);
    int *a = static_cast<int *>(std::malloc(bytes));
    int *b = static_cast<int *>(std::malloc(bytes));
    int *c = static_cast<int *>(std::malloc(bytes));
    int *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    if (!a || !b || !c) {
        std::fprintf(stderr, "Could not allocate host arrays.\n");
        std::free(a); std::free(b); std::free(c);
        return 1;
    }
    random_ints(a, N);
    random_ints(b, N);

    if (!check_cuda(cudaMalloc(&d_a, bytes), "cudaMalloc(d_a)") ||
        !check_cuda(cudaMalloc(&d_b, bytes), "cudaMalloc(d_b)") ||
        !check_cuda(cudaMalloc(&d_c, bytes), "cudaMalloc(d_c)"))
        return 1;
    if (!check_cuda(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice), "copy a to device") ||
        !check_cuda(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice), "copy b to device"))
        return 1;

    dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    if (!check_cuda(cudaGetLastError(), "add kernel launch")) return 1;
    if (!check_cuda(cudaDeviceSynchronize(), "add kernel execution")) return 1;
    dElaps = cpuSecond() - dStart;
    if (!check_cuda(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost), "copy result to host")) return 1;

    int errors = 0;
    for (int i = 0; i < N; ++i) {
        std::printf("%d + %d = %d\n", a[i], b[i], c[i]);
        if (c[i] != a[i] + b[i])
            ++errors;
    }
    std::printf("Verification: %d mismatches out of %d\n", errors, N);
    std::printf("Time taken - %.7f\n", dElaps);

    std::free(a); std::free(b); std::free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return errors == 0 ? 0 : 1;
}


In [ ]:
!nvcc -arch=sm_75  ex1_lab9_ex3_timed.cu -o lab9_ex3_timed && ./lab9_ex3_timed

In [ ]:
%%writefile ex1_lab9_ex4_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void addT(const int *a, const int *b, int *c)
{
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

static bool check_cuda(cudaError_t error, const char *operation)
{
    if (error == cudaSuccess)
        return true;
    std::fprintf(stderr, "%s failed: %s\n", operation, cudaGetErrorString(error));
    return false;
}

static void random_ints(int *values, int count)
{
    for (int i = 0; i < count; ++i)
        values[i] = std::rand() % 100;
}

static double cpuSecond()
{
    timeval time_value{};
    gettimeofday(&time_value, nullptr);
    return static_cast<double>(time_value.tv_sec) + time_value.tv_usec * 1.0e-6;
}

int main()
{
    double dStart = 0.0, dElaps = 0.0;
    const size_t bytes = static_cast<size_t>(N) * sizeof(int);
    int *a = static_cast<int *>(std::malloc(bytes));
    int *b = static_cast<int *>(std::malloc(bytes));
    int *c = static_cast<int *>(std::malloc(bytes));
    int *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    if (!a || !b || !c) {
        std::fprintf(stderr, "Could not allocate host arrays.\n");
        std::free(a); std::free(b); std::free(c);
        return 1;
    }
    random_ints(a, N);
    random_ints(b, N);

    if (!check_cuda(cudaMalloc(&d_a, bytes), "cudaMalloc(d_a)") ||
        !check_cuda(cudaMalloc(&d_b, bytes), "cudaMalloc(d_b)") ||
        !check_cuda(cudaMalloc(&d_c, bytes), "cudaMalloc(d_c)"))
        return 1;
    if (!check_cuda(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice), "copy a to device") ||
        !check_cuda(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice), "copy b to device"))
        return 1;

    dStart = cpuSecond();
    addT<<<1, N>>>(d_a, d_b, d_c);
    if (!check_cuda(cudaGetLastError(), "addT kernel launch")) return 1;
    if (!check_cuda(cudaDeviceSynchronize(), "addT kernel execution")) return 1;
    dElaps = cpuSecond() - dStart;
    if (!check_cuda(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost), "copy result to host")) return 1;

    int errors = 0;
    for (int i = 0; i < N; ++i) {
        std::printf("%d) %d + %d = %d\n", i, a[i], b[i], c[i]);
        if (c[i] != a[i] + b[i])
            ++errors;
    }
    std::printf("Verification: %d mismatches out of %d\n", errors, N);
    std::printf("Time taken - %.7f\n", dElaps);

    std::free(a); std::free(b); std::free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return errors == 0 ? 0 : 1;
}


In [ ]:
!nvcc -arch=sm_75  ex1_lab9_ex4_timed.cu -o lab9_ex4_timed && ./lab9_ex4_timed

In [ ]:
%%writefile ex1_lab9_ex5_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#include <cstddef>

#define N 10000000
#define THREADS_PER_BLOCK 512

__global__ void vecMul(const int *a, const int *b, int *c, int n)
{
    const int i = static_cast<int>(blockIdx.x * blockDim.x + threadIdx.x);
    if (i < n)
        c[i] = a[i] * b[i];
}

static bool check_cuda(cudaError_t error, const char *operation)
{
    if (error == cudaSuccess)
        return true;
    std::fprintf(stderr, "%s failed: %s\n", operation, cudaGetErrorString(error));
    return false;
}

static void random_ints(int *values, int count)
{
    for (int i = 0; i < count; ++i)
        values[i] = std::rand() % 100;
}

static double cpuSecond()
{
    timeval time_value{};
    gettimeofday(&time_value, nullptr);
    return static_cast<double>(time_value.tv_sec) + time_value.tv_usec * 1.0e-6;
}

int main()
{
    double dStart = 0.0, dElaps = 0.0;
    const size_t bytes = static_cast<size_t>(N) * sizeof(int);
    int *a = static_cast<int *>(std::malloc(bytes));
    int *b = static_cast<int *>(std::malloc(bytes));
    int *c = static_cast<int *>(std::malloc(bytes));
    int *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    if (!a || !b || !c) {
        std::fprintf(stderr, "Could not allocate the host vectors.\n");
        std::free(a); std::free(b); std::free(c);
        return 1;
    }
    random_ints(a, N);
    random_ints(b, N);

    if (!check_cuda(cudaMalloc(&d_a, bytes), "cudaMalloc(d_a)") ||
        !check_cuda(cudaMalloc(&d_b, bytes), "cudaMalloc(d_b)") ||
        !check_cuda(cudaMalloc(&d_c, bytes), "cudaMalloc(d_c)"))
        return 1;
    if (!check_cuda(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice), "copy a to device") ||
        !check_cuda(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice), "copy b to device"))
        return 1;

    const int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    std::printf("Launching %d blocks x %d threads\n", blocks, THREADS_PER_BLOCK);
    dStart = cpuSecond();
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    if (!check_cuda(cudaGetLastError(), "vecMul kernel launch")) return 1;
    if (!check_cuda(cudaDeviceSynchronize(), "vecMul kernel execution")) return 1;
    dElaps = cpuSecond() - dStart;
    if (!check_cuda(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost), "copy result to host")) return 1;

    for (int i = N - 1000; i < N; ++i)
        std::printf("%d) %d x %d = %d\n", i, a[i], b[i], c[i]);

    int errors = 0;
    for (int i = 0; i < N; ++i)
        if (c[i] != a[i] * b[i])
            ++errors;
    std::printf("Verification: %d mismatches out of %d\n", errors, N);
    std::printf("Time taken - %.7f\n", dElaps);

    std::free(a); std::free(b); std::free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return errors == 0 ? 0 : 1;
}


In [ ]:
!nvcc -arch=sm_75  ex1_lab9_ex5_timed.cu -o lab9_ex5_timed && ./lab9_ex5_timed

In [ ]:
%%writefile ex1_lab9_ex6_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#include <cstddef>

#define ROWS 10000
#define COLS 10000

__global__ void mulElem(const int *a, const int *b, int *c, int rows, int cols)
{
    const int col = static_cast<int>(blockIdx.x * blockDim.x + threadIdx.x);
    const int row = static_cast<int>(blockIdx.y * blockDim.y + threadIdx.y);
    if (row < rows && col < cols) {
        const int idx = row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

static bool check_cuda(cudaError_t error, const char *operation)
{
    if (error == cudaSuccess)
        return true;
    std::fprintf(stderr, "%s failed: %s\n", operation, cudaGetErrorString(error));
    return false;
}

static void random_ints(int *values, size_t count)
{
    for (size_t i = 0; i < count; ++i)
        values[i] = std::rand() % 100;
}

static double cpuSecond()
{
    timeval time_value{};
    gettimeofday(&time_value, nullptr);
    return static_cast<double>(time_value.tv_sec) + time_value.tv_usec * 1.0e-6;
}

int main()
{
    double dStart = 0.0, dElaps = 0.0;
    const size_t count = static_cast<size_t>(ROWS) * COLS;
    const size_t bytes = count * sizeof(int);
    int *a = static_cast<int *>(std::malloc(bytes));
    int *b = static_cast<int *>(std::malloc(bytes));
    int *c = static_cast<int *>(std::malloc(bytes));
    int *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    if (!a || !b || !c) {
        std::fprintf(stderr, "Could not allocate the host matrices.\n");
        std::free(a); std::free(b); std::free(c);
        return 1;
    }
    random_ints(a, count);
    random_ints(b, count);

    if (!check_cuda(cudaMalloc(&d_a, bytes), "cudaMalloc(d_a)") ||
        !check_cuda(cudaMalloc(&d_b, bytes), "cudaMalloc(d_b)") ||
        !check_cuda(cudaMalloc(&d_c, bytes), "cudaMalloc(d_c)"))
        return 1;
    if (!check_cuda(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice), "copy a to device") ||
        !check_cuda(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice), "copy b to device"))
        return 1;

    const dim3 threadsPerBlock(32, 16);
    const dim3 numBlocks((COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
                         (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y);
    std::printf("Grid: %u x %u blocks\n", numBlocks.x, numBlocks.y);
    dStart = cpuSecond();
    mulElem<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, ROWS, COLS);
    if (!check_cuda(cudaGetLastError(), "mulElem kernel launch")) return 1;
    if (!check_cuda(cudaDeviceSynchronize(), "mulElem kernel execution")) return 1;
    dElaps = cpuSecond() - dStart;
    if (!check_cuda(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost), "copy result to host")) return 1;

    const int last_row = ROWS - 1;
    for (int col = COLS - 1000; col < COLS; ++col) {
        const size_t idx = static_cast<size_t>(last_row) * COLS + col;
        std::printf("C[%d][%d] = %d x %d = %d\n", last_row, col, a[idx], b[idx], c[idx]);
    }

    int errors = 0;
    for (size_t i = 0; i < count; ++i)
        if (c[i] != a[i] * b[i])
            ++errors;
    std::printf("Verification: %d mismatches out of %zu\n", errors, count);
    std::printf("Time taken - %.7f\n", dElaps);

    std::free(a); std::free(b); std::free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return errors == 0 ? 0 : 1;
}


In [ ]:
!nvcc -arch=sm_75  ex1_lab9_ex6_timed.cu -o lab9_ex6_timed && ./lab9_ex6_timed

In [ ]:
import csv, re, statistics, subprocess
programs = [
    ("Lab 9 Exercise 3 (512 x 1)", "lab9_ex3_timed"),
    ("Lab 9 Exercise 4 (1 x 512)", "lab9_ex4_timed"),
    ("Lab 9 Exercise 5 (10M elements)", "lab9_ex5_timed"),
    ("Lab 9 Exercise 6 (100M elements)", "lab9_ex6_timed"),
]
rows = []
for label, executable in programs:
    for run_number in range(1, 4):
        result = subprocess.run(["./" + executable], check=True, capture_output=True, text=True)
        match = re.search(r"Time taken - ([0-9.eE+-]+)", result.stdout)
        if not match:
            raise RuntimeError(f"No kernel time found in {executable} output")
        rows.append((label, run_number, float(match.group(1))))
print(f"{'Program':42} {'Run 1 (s)':>12} {'Run 2 (s)':>12} {'Run 3 (s)':>12}")
for label, _ in programs:
    values = [seconds for name, _, seconds in rows if name == label]
    print(f"{label:42} " + " ".join(f"{value:12.7f}" for value in values))
with open("lab9_timing_results.csv", "w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(["program", "run", "kernel_seconds"])
    writer.writerows(rows)
print("Saved lab9_timing_results.csv")

**Compare the times:** Exercises 3 and 4 do the same 512 additions with different block layouts. The one-block version can avoid scheduling many tiny blocks, but these kernels are so small that launch overhead and GPU variation can dominate. Exercise 6 processes 10 times as many elements as Exercise 5; its kernel time need not be exactly 10 times as long because memory throughput, occupancy, and workload size affect utilization. Use your three measured values to support the comparison.

### Kernel time versus data-transfer time (100 million elements)

This copy changes Lab 9 Exercise 5 to 100 million elements. `Kernel time` covers only the kernel and synchronization. `Copy-in + kernel + copy-out time` starts before the first host-to-device copy and stops after the device-to-host copy. The comparison excludes memory allocation and random-number generation.

In [ ]:
%%writefile ex1_lab9_ex5_100m_transfer_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#include <cstddef>

#define N 100000000
#define THREADS_PER_BLOCK 512

__global__ void vecMul(const int *a, const int *b, int *c, int n)
{
    const int i = static_cast<int>(blockIdx.x * blockDim.x + threadIdx.x);
    if (i < n)
        c[i] = a[i] * b[i];
}

static bool check_cuda(cudaError_t error, const char *operation)
{
    if (error == cudaSuccess)
        return true;
    std::fprintf(stderr, "%s failed: %s\n", operation, cudaGetErrorString(error));
    return false;
}

static void random_ints(int *values, int count)
{
    for (int i = 0; i < count; ++i)
        values[i] = std::rand() % 100;
}

static double cpuSecond()
{
    timeval time_value{};
    gettimeofday(&time_value, nullptr);
    return static_cast<double>(time_value.tv_sec) + time_value.tv_usec * 1.0e-6;
}

int main()
{
    double dStart = 0.0, dElaps = 0.0, transferStart = 0.0, transferElapsed = 0.0;
    const size_t bytes = static_cast<size_t>(N) * sizeof(int);
    int *a = static_cast<int *>(std::malloc(bytes));
    int *b = static_cast<int *>(std::malloc(bytes));
    int *c = static_cast<int *>(std::malloc(bytes));
    int *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    if (!a || !b || !c) {
        std::fprintf(stderr, "Could not allocate the host vectors.\n");
        std::free(a); std::free(b); std::free(c);
        return 1;
    }
    random_ints(a, N);
    random_ints(b, N);

    if (!check_cuda(cudaMalloc(&d_a, bytes), "cudaMalloc(d_a)") ||
        !check_cuda(cudaMalloc(&d_b, bytes), "cudaMalloc(d_b)") ||
        !check_cuda(cudaMalloc(&d_c, bytes), "cudaMalloc(d_c)"))
        return 1;
    transferStart = cpuSecond();
    if (!check_cuda(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice), "copy a to device") ||
        !check_cuda(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice), "copy b to device"))
        return 1;

    const int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    std::printf("Launching %d blocks x %d threads\n", blocks, THREADS_PER_BLOCK);
    dStart = cpuSecond();
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    if (!check_cuda(cudaGetLastError(), "vecMul kernel launch")) return 1;
    if (!check_cuda(cudaDeviceSynchronize(), "vecMul kernel execution")) return 1;
    dElaps = cpuSecond() - dStart;
    if (!check_cuda(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost), "copy result to host")) return 1;
    transferElapsed = cpuSecond() - transferStart;

    for (int i = N - 1000; i < N; ++i)
        std::printf("%d) %d x %d = %d\n", i, a[i], b[i], c[i]);

    int errors = 0;
    for (int i = 0; i < N; ++i)
        if (c[i] != a[i] * b[i])
            ++errors;
    std::printf("Verification: %d mismatches out of %d\n", errors, N);
    std::printf("Kernel time - %.7f\n", dElaps);
    std::printf("Copy-in + kernel + copy-out time - %.7f\n", transferElapsed);

    std::free(a); std::free(b); std::free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return errors == 0 ? 0 : 1;
}


In [ ]:
!nvcc -arch=sm_75  ex1_lab9_ex5_100m_transfer_timed.cu -o ex5_100m_transfer && ./ex5_100m_transfer

The full sequence should take longer because it moves about 800 MB to the GPU and 400 MB back. The measured ratio depends on the assigned GPU and host-to-GPU bandwidth. This shows that GPU work benefits most when the computation is substantial relative to the cost of moving its data.

## Exercise 2 - Catch errors with the CHECK macro

The macro checks CUDA runtime calls, the kernel-launch status, and execution after synchronization. Run the normal program first; it should verify all 512 results and print a time.

In [ ]:
%%writefile ex2_check.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

#define CHECK(call) do { \
    const cudaError_t error = (call); \
    if (error != cudaSuccess) { \
        std::printf("Error: %s:%d, code:%d, reason: %s\n", \
                    __FILE__, __LINE__, static_cast<int>(error), cudaGetErrorString(error)); \
        std::exit(1); \
    } \
} while (0)

__global__ void add(const int *a, const int *b, int *c)
{
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

static double cpuSecond()
{
    timeval time_value{};
    gettimeofday(&time_value, nullptr);
    return static_cast<double>(time_value.tv_sec) + time_value.tv_usec * 1.0e-6;
}

static void random_ints(int *values, int count)
{
    for (int i = 0; i < count; ++i)
        values[i] = std::rand() % 100;
}

int main()
{
    int *a = nullptr, *b = nullptr, *c = nullptr;
    int *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    const size_t bytes = static_cast<size_t>(N) * sizeof(int);
    double started, elapsed;

#ifdef EXPERIMENT_BAD_ALLOC
    CHECK(cudaMalloc(&d_a, static_cast<size_t>(1) << 50));
#else
    CHECK(cudaMalloc(&d_a, bytes));
#endif
    CHECK(cudaMalloc(&d_b, bytes));
    CHECK(cudaMalloc(&d_c, bytes));
    a = static_cast<int *>(std::malloc(bytes));
    b = static_cast<int *>(std::malloc(bytes));
    c = static_cast<int *>(std::malloc(bytes));
    if (!a || !b || !c) {
        std::fprintf(stderr, "Could not allocate host arrays.\n");
        return 1;
    }
    random_ints(a, N);
    random_ints(b, N);
    CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));

    started = cpuSecond();
#ifdef EXPERIMENT_BAD_LAUNCH
    add<<<N, 2048>>>(d_a, d_b, d_c);
#else
    add<<<N, 1>>>(d_a, d_b, d_c);
#endif
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    elapsed = cpuSecond() - started;
    CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));

    for (int i = 0; i < 5; ++i)
        std::printf("%d + %d = %d\n", a[i], b[i], c[i]);
    int errors = 0;
    for (int i = 0; i < N; ++i)
        if (c[i] != a[i] + b[i])
            ++errors;
    std::printf("Verification: %d mismatches out of %d\n", errors, N);
    std::printf("Time taken - %.7f\n", elapsed);

    std::free(a); std::free(b); std::free(c);
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
    return errors == 0 ? 0 : 1;
}


In [ ]:
!nvcc -arch=sm_75  ex2_check.cu -o ex2_check && ./ex2_check

### Break it on purpose

Experiment A uses 2,048 threads in one block. Experiment B requests one petabyte. `CHECK` should report the failing file, line, code, and reason. These runs are expected to exit with an error, so the shell cells keep the notebook running. Re-run the normal cell above to restore the valid program.

In [ ]:
!nvcc -arch=sm_75 -DEXPERIMENT_BAD_LAUNCH ex2_check.cu -o ex2_bad_launch && (./ex2_bad_launch || true)

In [ ]:
!nvcc -arch=sm_75 -DEXPERIMENT_BAD_ALLOC ex2_check.cu -o ex2_bad_alloc && (./ex2_bad_alloc || true)

**Experiment C - no CHECK:** the plain Exercise 1 program does not wrap the calls in `CHECK`. Compile it with the bad launch below. Its verification should reveal wrong or missing results, but without checking the launch and copy return values the failure is harder to locate.

In [ ]:
!nvcc -arch=sm_75 -DEXPERIMENT_BAD_LAUNCH ex1_timed.cu -o ex1_bad_launch && (./ex1_bad_launch || true)

The `do { ... } while (0)` wrapper makes `CHECK(call);` behave like one ordinary statement, including in an unbraced `if` statement.

## Exercise 3 - Scale a picture with a 2D grid

Each pixel is a flat-array `unsigned char`. A 16 × 16 block maps x to columns and y to rows. The kernel doubles each value and saturates at 255. The CPU check applies the same cap, prints the original and updated top-left 50 × 50 portions, and counts capped pixels.

In [ ]:
%%writefile ex3_picture.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define WIDTH 800
#define HEIGHT 600

#define CHECK(call) do { \
    const cudaError_t error = (call); \
    if (error != cudaSuccess) { \
        std::printf("Error: %s:%d, code:%d, reason: %s\n", \
                    __FILE__, __LINE__, static_cast<int>(error), cudaGetErrorString(error)); \
        std::exit(1); \
    } \
} while (0)

static double cpuSecond()
{
    timeval time_value{};
    gettimeofday(&time_value, nullptr);
    return static_cast<double>(time_value.tv_sec) + time_value.tv_usec * 1.0e-6;
}

__global__ void PictureKernel(const unsigned char *input, unsigned char *output,
                              int height, int width)
{
    const int row = blockIdx.y * blockDim.y + threadIdx.y;
    const int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < height && col < width) {
        const int index = row * width + col;
        const int scaled = 2 * input[index];
        output[index] = static_cast<unsigned char>(scaled > 255 ? 255 : scaled);
    }
}

int main()
{
    const size_t count = static_cast<size_t>(WIDTH) * HEIGHT;
    const size_t bytes = count * sizeof(unsigned char);
    auto *input = static_cast<unsigned char *>(std::malloc(bytes));
    auto *output = static_cast<unsigned char *>(std::malloc(bytes));
    unsigned char *d_input = nullptr, *d_output = nullptr;
    if (!input || !output) {
        std::fprintf(stderr, "Could not allocate host pixel arrays.\n");
        std::free(input); std::free(output);
        return 1;
    }
    for (size_t i = 0; i < count; ++i)
        input[i] = static_cast<unsigned char>(std::rand() % 256);

    CHECK(cudaMalloc(&d_input, bytes));
    CHECK(cudaMalloc(&d_output, bytes));
    CHECK(cudaMemcpy(d_input, input, bytes, cudaMemcpyHostToDevice));

    const dim3 threadsPerBlock(16, 16);
    const dim3 numBlocks((WIDTH + threadsPerBlock.x - 1) / threadsPerBlock.x,
                         (HEIGHT + threadsPerBlock.y - 1) / threadsPerBlock.y);
    std::printf("Grid: %u x %u blocks of %u x %u threads\n",
                numBlocks.x, numBlocks.y, threadsPerBlock.x, threadsPerBlock.y);

    const double started = cpuSecond();
    PictureKernel<<<numBlocks, threadsPerBlock>>>(d_input, d_output, HEIGHT, WIDTH);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    const double elapsed = cpuSecond() - started;
    CHECK(cudaMemcpy(output, d_output, bytes, cudaMemcpyDeviceToHost));

    std::printf("Original top-left 50 x 50:\n");
    for (int row = 0; row < 50; ++row) {
        for (int col = 0; col < 50; ++col)
            std::printf("%3u ", static_cast<unsigned int>(input[row * WIDTH + col]));
        std::printf("\n");
    }
    std::printf("Updated top-left 50 x 50:\n");
    for (int row = 0; row < 50; ++row) {
        for (int col = 0; col < 50; ++col)
            std::printf("%3u ", static_cast<unsigned int>(output[row * WIDTH + col]));
        std::printf("\n");
    }

    size_t capped = 0;
    size_t mismatches = 0;
    for (size_t i = 0; i < count; ++i) {
        const int scaled = 2 * static_cast<int>(input[i]);
        const unsigned char expected = static_cast<unsigned char>(scaled > 255 ? 255 : scaled);
        if (scaled > 255)
            ++capped;
        if (output[i] != expected)
            ++mismatches;
    }
    std::printf("Verification: %zu mismatches out of %zu\n", mismatches, count);
    std::printf("Pixels capped at 255: %zu\n", capped);
    std::printf("Time taken - %.7f\n", elapsed);

    std::free(input); std::free(output);
    CHECK(cudaFree(d_input)); CHECK(cudaFree(d_output));
    return mismatches == 0 ? 0 : 1;
}


In [ ]:
!nvcc -arch=sm_75  ex3_picture.cu -o ex3_picture && ./ex3_picture

The expected grid is 50 × 38 blocks, or 486,400 total threads. The last block row has 8 active rows and 8 out-of-range rows. Values 128–255 become 255 after doubling and saturation, so about half of the 480,000 pixels are capped (the handout’s sample count is 240,121). Verify the count from your Colab run.

## Exercise 4 - Multiply a 500 × 500 × 500 volume element by element

A 3D block has 8 × 8 × 8 = 512 threads. Each thread maps x, y, and z to one volume element and checks all three bounds before using the row-major flat index `(z * NY + y) * NX + x`. The timer covers the kernel only. The program prints the last 1,000 results and checks all 125,000,000 values on the CPU.

In [ ]:
%%writefile ex4_volume.cu
#include <cstdio>
#include <cstdlib>
#include <cstddef>
#include <sys/time.h>

#define NX 500
#define NY 500
#define NZ 500

#define CHECK(call) do { \
    const cudaError_t error = (call); \
    if (error != cudaSuccess) { \
        std::printf("Error: %s:%d, code:%d, reason: %s\n", \
                    __FILE__, __LINE__, static_cast<int>(error), cudaGetErrorString(error)); \
        std::exit(1); \
    } \
} while (0)

static double cpuSecond()
{
    timeval time_value{};
    gettimeofday(&time_value, nullptr);
    return static_cast<double>(time_value.tv_sec) + time_value.tv_usec * 1.0e-6;
}

__global__ void mulElem3D(const int *a, const int *b, int *c,
                          int nx, int ny, int nz)
{
    const int x = blockIdx.x * blockDim.x + threadIdx.x;
    const int y = blockIdx.y * blockDim.y + threadIdx.y;
    const int z = blockIdx.z * blockDim.z + threadIdx.z;
    if (x < nx && y < ny && z < nz) {
        const size_t index = (static_cast<size_t>(z) * ny + y) * nx + x;
        c[index] = a[index] * b[index];
    }
}

static void random_ints(int *values, size_t count)
{
    for (size_t i = 0; i < count; ++i)
        values[i] = std::rand() % 100;
}

int main()
{
    const size_t count = static_cast<size_t>(NX) * NY * NZ;
    const size_t bytes = count * sizeof(int);
    int *a = static_cast<int *>(std::malloc(bytes));
    int *b = static_cast<int *>(std::malloc(bytes));
    int *c = static_cast<int *>(std::malloc(bytes));
    int *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    if (!a || !b || !c) {
        std::fprintf(stderr, "Could not allocate the host volumes.\n");
        std::free(a); std::free(b); std::free(c);
        return 1;
    }
    random_ints(a, count);
    random_ints(b, count);

    CHECK(cudaMalloc(&d_a, bytes));
    CHECK(cudaMalloc(&d_b, bytes));
    CHECK(cudaMalloc(&d_c, bytes));
    CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));

    const dim3 threadsPerBlock(8, 8, 8);
    const dim3 numBlocks((NX + threadsPerBlock.x - 1) / threadsPerBlock.x,
                         (NY + threadsPerBlock.y - 1) / threadsPerBlock.y,
                         (NZ + threadsPerBlock.z - 1) / threadsPerBlock.z);
    std::printf("Grid: %u x %u x %u blocks\n", numBlocks.x, numBlocks.y, numBlocks.z);

    const double started = cpuSecond();
    mulElem3D<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, NX, NY, NZ);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    const double elapsed = cpuSecond() - started;
    CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));

    for (size_t index = count - 1000; index < count; ++index) {
        const size_t x = index % NX;
        const size_t y = (index / NX) % NY;
        const size_t z = index / (static_cast<size_t>(NX) * NY);
        std::printf("C[%zu][%zu][%zu] = %d x %d = %d\n",
                    z, y, x, a[index], b[index], c[index]);
    }

    size_t errors = 0;
    for (size_t i = 0; i < count; ++i)
        if (c[i] != a[i] * b[i])
            ++errors;
    std::printf("Verification: %zu mismatches out of %zu\n", errors, count);
    std::printf("Time taken - %.7f\n", elapsed);

    std::free(a); std::free(b); std::free(c);
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
    return errors == 0 ? 0 : 1;
}


In [ ]:
!nvcc -arch=sm_75  ex4_volume.cu -o ex4_volume && ./ex4_volume

The expected grid is 63 × 63 × 63 blocks (250,047 blocks). The 3D version has 25% more elements than Lab 9 Exercise 6. Compare its measured kernel time with the Lab 9 2D result: the two layouts do similar per-element work, so memory access and occupancy matter more than the number of dimensions by itself. The 3D random input generation can take several seconds, but it is outside the kernel timer.

**Before leaving Colab:** save this notebook to Drive, then disconnect and delete the runtime to release the GPU.